In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
%matplotlib widget
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch
import torch.optim as optim
import torch.nn as nn
from core.benchmarks import *
from core.CardiacCTdataset import DataLoaderFactory
from core.CNNmodel import *
from core.benchmarks import *
import pandas as pd
from core.CVsplits import *
from tqdm.notebook import tqdm
import logging
from core.Log import *
setup_loggers5K()
OUTER_FOLDS = 5; INNER_FOLDS = 3


C:\Users\sulei\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\requests\__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


In [ ]:
# Create Holdout dataset for FINAL Model Evaluation
from sklearn.model_selection import train_test_split
from core.Log import load_dataset_info, save_dataset_info


main_dataset = load_dataset_info(file="data/data_info.json")
labels = [lbl['label'] for lbl in main_dataset]
main_training, final_test = train_test_split(main_dataset,
											 test_size=22,     # 33 for 4 OUTER folds, 22 for 5 OUTER folds
											 stratify=labels,
											 random_state=42)  # 67 FOR 5 OUTER FOLDS
print(len(final_test))

for sample in main_dataset:
	if sample in final_test: sample['pool'] = 'holdout'
	else: sample['pool'] = 'main'
#save_dataset_info(main_dataset, file="NCV_5_3_folds/data_info_5-3NCV.json")
#"data/data_info_5-3NCV.json"


22
Successfully saved data to: NCV_5_3_folds/data_info_5-3NCV.json


In [8]:
from core.CVsplits import create_folds_stats

create_folds_stats(OUTER_K=5, INNER_K=3)


Loaded NCV_5_3_folds/data_info_5-3NCV.json.
Generating and saving fold indices...
OUTER FOLD 0 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 1 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 2 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 3 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER

In [ ]:
import itertools
import json
## 1. Define all model-specific hyperparameter sweeps in one dictionary
model_configs = {
	"MultiViewCNN": {
		"LR_SWEEP": [5e-4, 1e-4],
		"DR_SWEEP": [0.35],
		"WD_SWEEP": [1e-6, 1e-4]
	},
	"META+MLP": {
		"LR_SWEEP": [3e-4, 1e-3],
		"DR_SWEEP": [0.4],
		"WD_SWEEP": [1e-6, 1e-4]
	},
	"RN18+MLP": {
		"LR_SWEEP": [1e-3, 5e-4],
		"DR_SWEEP": [0.4],
		"WD_SWEEP": [5e-4]
	},
	"Axial":    {"LR_SWEEP": [3e-4, 1e-3], "WD_SWEEP": [1e-6, 1e-4], "DR_SWEEP": [0.35]},
	"Coronal":  {"LR_SWEEP": [3e-4], "WD_SWEEP": [1e-4], "DR_SWEEP": [0.35]},
	"Sagittal": {"LR_SWEEP": [3e-4], "WD_SWEEP": [1e-4], "DR_SWEEP": [0.35]},
}

# 2. Define global parameters that are the same for all models
GLOBAL_PARAMS = {
	"P": 5,
	"Epochs": 30,
}

INNER_CV_parameters = []
ID = 1

# Iterate through each model and its specific configuration
for model_name, config in model_configs.items():

	# Generate all unique combinations of the model's hyperparameters
	# e.g., for MultiViewCNN, this will create (1e-3, 0.3, 1e-4), (1e-3, 0.4, 1e-4), etc.
	hp_combinations = list(itertools.product(
		config['LR_SWEEP'],
		config['DR_SWEEP'],
		config['WD_SWEEP']
	))

	# Loop through outer and inner folds
	for outer_fold_idx in range(0, 5):
		for inner_fold_idx in range(0, 3):
			# Loop through each hyperparameter combination for this model
			for i, (lr, dr, wd) in enumerate(hp_combinations):
				item = {
					"ExpID": ID,
					"Model": model_name,
					'OUTER_FOLD': outer_fold_idx,
					'INNER_FOLD': inner_fold_idx,
					"hypers": {
						"HPset": i + 1,
						"LR": lr,
						"WD": wd,
						"DR": dr,
						"P": GLOBAL_PARAMS['P'],
						"Epochs": GLOBAL_PARAMS['Epochs'],
					},
					"trained": False
				}
				INNER_CV_parameters.append(item)
				ID += 1

print(f"Total combinations generated: {len(INNER_CV_parameters)}")

#with open("NCV_5_3_folds/INNER_experiments.json", "w") as f:
#	json.dump(INNER_CV_parameters, f, indent=2)



Total combinations generated: 240


In [3]:
INNER_CV_parameters = load_from_json("NCV_5_3_folds/INNER_experiments.json")
df = pd.DataFrame(INNER_CV_parameters)
#df.to_csv("INNER_experiments.csv", index=False)


Loaded NCV_5_3_folds/INNER_experiments.json.


In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, roc_curve, f1_score

def append_experiment_results(item, path="NCV_5_3_folds/INNER_results.jsonl"):
	with open(path, "a") as f:  # append mode
		f.write(json.dumps(item) + "\n")

def best_threshold(all_labels, all_probs, utility="youden"):
	"""
	Pick a post-hoc decision threshold on validation predictions.
	utility: "youden" (maximize TPR-FPR) or "f1".
	"""
	if utility == "f1":
		# scan unique probabilities for F1
		# (for speed you can sample a subset if very large)
		thr = np.unique(all_probs)
		f1s = [f1_score(all_labels, all_probs >= t) for t in thr]
		idx = int(np.argmax(f1s))
		return float(thr[idx])
	else:
		fpr, tpr, thr = roc_curve(all_labels, all_probs)
		j = tpr - fpr
		idx = int(np.argmax(j))
		return float(thr[idx])  # may be outside [0,1] if degenerate; fine.



def train_INNER_model(model, train_loader, val_loader, experiment):
	log = logging.getLogger('INNER_5Ktrain')
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	model.to(device)
	hypers = experiment['hypers']
	ExpID = experiment['ExpID']
	epochs = hypers['Epochs']

	LR = hypers['LR']
	WD = hypers['WD']
	P = hypers['P']

	optimizer = optim.Adam(model.parameters(), lr= LR, weight_decay=WD)
	scheduler = ReduceLROnPlateau(optimizer, mode='min',
								  patience=2, factor=0.5,
								  threshold=1e-3, threshold_mode='rel',
								  cooldown=0, min_lr=1e-6)
	criterion = nn.BCEWithLogitsLoss()

	best_auc = -np.inf
	best_loss_at_best_auc = np.inf
	best_th = 0.5

	no_improve = 0

	val_N=len(val_loader.dataset)
	train_N=len(train_loader.dataset)


	print(f"	↳ Experiment {ExpID} | Training model... ")
	for epoch in range(epochs):
		model.train()
		running_loss = 0.0

		for batch in train_loader:
			axi = batch["axial_image"].to(device)
			cor = batch["coronal_image"].to(device)
			sag = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).unsqueeze(1)

			optimizer.zero_grad(set_to_none=True)

			logits = model(axi, sag, cor, met)
			T_loss = criterion(logits, lbl)

			T_loss.backward()
			optimizer.step()

			running_loss += T_loss.item() * lbl.size(0)

		T_loss = running_loss / train_N

		model.eval()
		running_loss = 0.0
		all_labels = []  #y_true
		all_probs = []   #y_pred

		with torch.no_grad():
			for batch in val_loader:
				axi = batch["axial_image"].to(device)
				cor = batch["coronal_image"].to(device)
				sag = batch["sagittal_image"].to(device)
				met = batch["meta"].to(device)
				lbl = batch["label"].to(device).unsqueeze(1)

				logits = model(axi, sag, cor, met)
				V_loss = criterion(logits, lbl)
				running_loss += V_loss.item() * lbl.size(0)

				all_labels.extend(lbl.cpu())
				all_probs.extend(torch.sigmoid(logits).cpu())

		V_loss = running_loss / val_N
		all_labels = torch.cat(all_labels).numpy().reshape(-1)
		all_probs = torch.cat(all_probs).numpy().reshape(-1)
		val_auc = roc_auc_score(all_labels, all_probs)

		scheduler.step(V_loss)
		th_star = best_threshold(all_labels, all_probs)
		improved = val_auc > best_auc + 1e-6

		log.info(f"{experiment['Model']};    {ExpID};    {experiment['OUTER_FOLD']};    {experiment['INNER_FOLD']};    {hypers['HPset']};    {epoch:02d};    {T_loss:.4f};    {V_loss:.4f}    {val_auc:.4f};    {th_star:.6f};    {optimizer.param_groups[0]['lr']};    {no_improve:02d};")

		if improved:
			best_auc = val_auc
			best_loss_at_best_auc = V_loss
			best_th = th_star
			best_epoch = epoch
			no_improve = 0
		else:
			no_improve += 1
		if no_improve >= 3: break

	results = {
		"ExpID": ExpID,
		"Model": experiment['Model'],
		"best_val_auc": float(best_auc),
		"best_val_loss": float(best_loss_at_best_auc),
		"best_threshold": float(best_th),
		"best_epoch": int(best_epoch),
		"epochs_ran": int(epoch)}

	return results


In [ ]:

log = logging.getLogger('INNER_5Ktrain')
full_dl = load_dataset_info(file="NCV_5_3_folds/data_info_5-3NCV.json")
main_dataset = [i for i in full_dl if i["pool"] == 'main']
#print(len(main_dataset))
DL = DataLoaderFactory(main_dataset)

INNER_CV_parameters = load_from_json("NCV_5_3_folds/INNER_experiments.json")
filtered = [
	exp for exp in INNER_CV_parameters
	if exp["Model"] == "MultiViewCNN"
	and exp["OUTER_FOLD"] == 2       # 2, 3, 4
	#and exp["hypers"]["HPset"] == 4
	and exp["trained"] == False
]

print(f"experiments to do: {len(filtered)}")
#log.info(f"Model;    ExpID;    OUTER_FOLD;    INNER_FOLD;    HPset;    epoch;    T_loss;    V_loss;    val_auc;    th_star;    LR;    no_improve;")
for experiment in filtered:
	print(experiment)
	OUT = experiment['OUTER_FOLD']
	INN = experiment['INNER_FOLD']
	train_loader, val_loader = DL.create_inner_loaders(OUT, INN)
	DR = experiment['hypers']['DR']
	model = MetadataMLP(DR)
	results = train_INNER_model(model, train_loader, val_loader, experiment)
	append_experiment_results(results, path="NCV_5_3_folds/INNER_results.jsonl")
	experiment["trained"] = True

save_to_json(INNER_CV_parameters, "NCV_5_3_folds/INNER_experiments.json")




Loaded NCV_5_3_folds/INNER_experiments.json.
experiments to do: 12
{'ExpID': 25, 'Model': 'MultiViewCNN', 'OUTER_FOLD': 2, 'INNER_FOLD': 0, 'hypers': {'HPset': 1, 'LR': 0.0005, 'WD': 1e-06, 'DR': 0.35, 'P': 5, 'Epochs': 30}, 'trained': False}
	↳ Experiment 25 | Training model... 
{'ExpID': 26, 'Model': 'MultiViewCNN', 'OUTER_FOLD': 2, 'INNER_FOLD': 0, 'hypers': {'HPset': 2, 'LR': 0.0005, 'WD': 0.0001, 'DR': 0.35, 'P': 5, 'Epochs': 30}, 'trained': False}
	↳ Experiment 26 | Training model... 
{'ExpID': 27, 'Model': 'MultiViewCNN', 'OUTER_FOLD': 2, 'INNER_FOLD': 0, 'hypers': {'HPset': 3, 'LR': 0.0001, 'WD': 1e-06, 'DR': 0.35, 'P': 5, 'Epochs': 30}, 'trained': False}
	↳ Experiment 27 | Training model... 
{'ExpID': 28, 'Model': 'MultiViewCNN', 'OUTER_FOLD': 2, 'INNER_FOLD': 0, 'hypers': {'HPset': 4, 'LR': 0.0001, 'WD': 0.0001, 'DR': 0.35, 'P': 5, 'Epochs': 30}, 'trained': False}
	↳ Experiment 28 | Training model... 
{'ExpID': 29, 'Model': 'MultiViewCNN', 'OUTER_FOLD': 2, 'INNER_FOLD': 1, 'h

KeyboardInterrupt: 

In [ ]:
def load_experiments_results(path="NCV_5_3_folds/INNER_results.jsonl"):
	experiments = []
	with open(path, "r") as f:
		for line in f:
			if line.strip():
				experiments.append(json.loads(line))
	return experiments


experiments_results = load_experiments_results(path="NCV_5_3_folds/INNER_results.jsonl")
experiments_results


[{'ExpID': 1,
  'Model': 'MultiViewCNN',
  'best_val_auc': 0.7368421052631579,
  'best_val_loss': 0.676205058892568,
  'best_threshold': 0.5266620516777039,
  'best_epoch': 1,
  'epochs_ran': 6},
 {'ExpID': 5,
  'Model': 'MultiViewCNN',
  'best_val_auc': 0.8699690402476781,
  'best_val_loss': 0.5614009963141547,
  'best_threshold': 0.4169628918170929,
  'best_epoch': 15,
  'epochs_ran': 20},
 {'ExpID': 9,
  'Model': 'MultiViewCNN',
  'best_val_auc': 0.736842105263158,
  'best_val_loss': 0.6903906530804105,
  'best_threshold': 0.48787832260131836,
  'best_epoch': 0,
  'epochs_ran': 5},
 {'ExpID': 2,
  'Model': 'MultiViewCNN',
  'best_val_auc': 0.7368421052631579,
  'best_val_loss': 0.6706827282905579,
  'best_threshold': 0.5295401215553284,
  'best_epoch': 2,
  'epochs_ran': 7},
 {'ExpID': 6,
  'Model': 'MultiViewCNN',
  'best_val_auc': 0.8452012383900929,
  'best_val_loss': 0.6643164820141263,
  'best_threshold': 0.5072835087776184,
  'best_epoch': 2,
  'epochs_ran': 7},
 {'ExpID': 10,

In [71]:
INNER_CV_parameters = load_from_json("NCV_5_3_folds/INNER_experiments.json")
experiments_results = load_experiments_results(path="NCV_5_3_folds/INNER_results.jsonl")
print(f"Total experiments: {len(INNER_CV_parameters)}, Total results: {len(experiments_results)}")



Loaded NCV_5_3_folds/INNER_experiments.json.
Total experiments: 240, Total results: 240


In [ ]:
def merge_experiment_data1(parameters, experiments_results):
	# 1) Build a fast lookup from results: ExpID -> result dict
	results_by_id = {}
	for r in experiments_results:
		exp_id = r.get("ExpID")
		if exp_id is not None and exp_id not in results_by_id:
			results_by_id[exp_id] = r  # keep first; adjust if you want "last wins"

	merged_data = []

	for param_obj in parameters:
		ExpID = param_obj["ExpID"]
		trained = param_obj.get("trained", False)
		completed = param_obj.get("completed", False)

		# 2) Try to find a result for this ExpID
		res = results_by_id.get(ExpID)

		if completed and trained:
			# Already done; still append
			merged_data.append(param_obj)
			continue

		if res is not None:
			# We have results; merge into the param object
			result_obj = dict(res)  # copy so we can modify safely
			result_obj.pop("ExpID", None)
			result_obj.pop("Model", None)
			param_obj.update(result_obj)

			# If results exist, ensure flags reflect completion
			param_obj["completed"] = True
			param_obj["trained"] = True  # flip to True if it was False
		else:
			# No results found
			if trained:
				# Trained but missing results → keep completed False
				param_obj["completed"] = False
			else:
				# Not trained and no results
				param_obj["completed"] = False

		merged_data.append(param_obj)

	return merged_data


INNER_CV_parameters = load_from_json("NCV_5_3_folds/INNER_experiments.json")
experiments_results = load_experiments_results(path="NCV_5_3_folds/INNER_results.jsonl")
merged_data = merge_experiment_data1(INNER_CV_parameters, experiments_results)
save_to_json(merged_data, "NCV_5_3_folds/INNER_experiments.json")



Loaded NCV_5_3_folds/INNER_experiments.json.


In [ ]:
def summarize_inner_cv(df, metric='best_val_auc'):
	"""
	Summarize inner-CV results per (Model, OUTER_FOLD, HPsetID) and select the best HPset.
	Returns (summary_df, winners_df).
	"""

	# keep only rows that have the metric
	dfm = df.dropna(subset=[metric]).copy()

	# aggregate per HPset across INNER_FOLDs
	summary = (
		dfm
		.groupby(['Model', 'OUTER_FOLD', 'HPsetID'], dropna=False)
		.agg(
			mean_metric=(metric, 'mean'),
			std_metric=(metric, 'std'),
			n_inner=('INNER_FOLD', 'nunique'),
			mean_val_loss=('best_val_loss', 'mean'),
			std_val_loss=('best_val_loss', 'std'),
			mean_threshold=('best_threshold', 'mean'),
			std_threshold=('best_threshold', 'std'),
		)
		.reset_index()
		.sort_values(['Model', 'OUTER_FOLD', 'HPsetID'])
	)
	return summary

INNER_CV_parameters = load_from_json("NCV_5_3_folds/INNER_experiments.json")
INNER_CV_parameters_df = pd.DataFrame(INNER_CV_parameters)
summary = summarize_inner_cv(INNER_CV_parameters_df, metric='best_val_auc')




Loaded NCV_5_3_folds/INNER_experiments.json.


In [ ]:

# MultiViewCNN fold 0: HPset 4
# MultiViewCNN fold 1: HPset 1
# MultiViewCNN fold 2: HPset 1
# MultiViewCNN fold 3: HPset 1
# MultiViewCNN fold 4: HPset 2

# RN18+MLP fold 0: HPset 2
# RN18+MLP fold 1: HPset 2
# RN18+MLP fold 2: HPset 2
# RN18+MLP fold 3: HPset 2
# RN18+MLP fold 4: HPset 1

# META+MLP fold 0: HPset 2
# META+MLP fold 1: HPset 3
# META+MLP fold 2: HPset 1
# META+MLP fold 3: HPset 3
# META+MLP fold 4: HPset 3

# Axial fold 0: HPset 4
# Axial fold 1: HPset 4
# Axial fold 2: HPset 3
# Axial fold 3: HPset 2
# Axial fold 4: HPset 4

# Sagittal fold 0: HPset 1
# Sagittal fold 1: HPset 1
# Sagittal fold 2: HPset 1
# Sagittal fold 3: HPset 1
# Sagittal fold 4: HPset 1

# Coronal fold 0: HPset 1
# Coronal fold 1: HPset 1
# Coronal fold 2: HPset 1
# Coronal fold 3: HPset 1
# Coronal fold 4: HPset 1


,Model,OUTER_FOLD,HPsetID,mean_metric,std_metric,n_inner,mean_val_loss,std_val_loss,mean_threshold,std_threshold
45,MultiViewCNN,0,1,0.781218,0.076861,3,0.642666,0.070734,0.477168,0.055628
46,MultiViewCNN,0,2,0.792570,0.054246,3,0.642123,0.044069,0.515158,0.012474
47,MultiViewCNN,0,3,0.799794,0.100718,3,0.677215,0.012160,0.530519,0.041641
48,MultiViewCNN,0,4,0.802890,0.072298,3,0.680569,0.013552,0.507963,0.025546
49,MultiViewCNN,1,1,0.899897,0.033104,3,0.593886,0.078328,0.410129,0.089116
50,MultiViewCNN,1,2,0.872033,0.061088,3,0.582073,0.091888,0.392204,0.199487
51,MultiViewCNN,1,3,0.820433,0.012384,3,0.671501,0.003238,0.533000,0.003486
52,MultiViewCNN,1,4,0.831785,0.031319,3,0.653056,0.028794,0.551172,0.033639
53,MultiViewCNN,2,1,0.914345,0.026331,3,0.584612,0.073723,0.551703,0.116486
54,MultiViewCNN,2,2,0.888545,0.044651,3,0.648820,0.031836,0.537071,0.025716


In [ ]:
import pandas as pd
from collections import Counter

# --- 1) winner selection from the summary produced by summarize_inner_cv ---
def pick_winners(summary: pd.DataFrame, higher_is_better: bool = True) -> pd.DataFrame:
	if summary.empty:
		return summary.copy()

	ascending_metric = not higher_is_better
	# sort by selection keys, then take head(1) per (Model, OUTER_FOLD)
	sort_keys = ['Model', 'OUTER_FOLD', 'mean_metric']
	sort_asc  = [True,     True,           ascending_metric]

	if 'mean_val_loss' in summary.columns:
		sort_keys += ['mean_val_loss']
		sort_asc  += [True]  # lower is better

	if 'std_metric' in summary.columns:
		sort_keys += ['std_metric']
		sort_asc  += [True]  # lower is better

	sort_keys += ['HPsetID']
	sort_asc  += [True]

	ranked = summary.sort_values(sort_keys, ascending=sort_asc)
	winners = (
		ranked
		.groupby(['Model', 'OUTER_FOLD'], as_index=False, sort=False)
		.head(1)
		.reset_index(drop=True)
	)
	return winners


# --- 2) helpers to extract a single hypers dict per (Model, OUTER_FOLD, HPsetID) ---
def _coalesce_mode(values):
	"""Return the most common non-null value, else None."""
	vals = [v for v in values if pd.notna(v)]
	if not vals:
		return None
	return Counter(vals).most_common(1)[0][0]

def _extract_hypers_for_hpset(inner_df: pd.DataFrame, model: str, outer_fold: int, hpset_id) -> dict:
	"""
	Find all INNER experiments matching (Model, OUTER_FOLD, HPsetID) or (hypers['HPset'] == HPsetID).
	From those rows, coalesce fields LR/WD/DR/P/Epochs into a single hypers dict.
	"""
	dfm = inner_df.copy()
	# Ensure HPsetID column exists even if it was only inside hypers
	if 'HPsetID' not in dfm.columns:
		if 'hypers' in dfm.columns:
			dfm['HPsetID'] = dfm['hypers'].apply(lambda h: (h or {}).get('HPset') if isinstance(h, dict) else pd.NA)
		else:
			dfm['HPsetID'] = pd.NA

	cand = dfm[(dfm['Model'] == model) & (dfm['OUTER_FOLD'] == outer_fold)]
	# match either by explicit HPsetID or by hypers['HPset']
	cand = cand[(cand['HPsetID'] == hpset_id) | (
		cand.get('hypers') is not None and
		cand['hypers'].apply(lambda h: isinstance(h, dict) and h.get('HPset') == hpset_id)
	)]

	# pull fields from 'hypers' dict; coalesce if multiple rows
	def pull(field, default=None):
		vals = []
		for _, r in cand.iterrows():
			h = r.get('hypers', {}) if isinstance(r.get('hypers'), dict) else {}
			vals.append(h.get(field, default))
		return _coalesce_mode(vals)

	hypers = {
		'HPset':   hpset_id,
		'LR':      pull('LR'),
		'WD':      pull('WD'),
		'DR':      pull('DR'),
		'TH':      None,  # set later from summary mean_threshold
		'P':       pull('P', 5),
		'Epochs':  50,
	}
	return hypers


# --- 3) main constructor: OUTER experiments list ---
def build_outer_experiments(summary_df: pd.DataFrame,
							inner_params_records: list,
							higher_is_better: bool = True,
							threshold_source: str = 'mean_threshold',
							start_exp_id: int = 1) -> list:
	"""
	From the summary (one row per Model×OUTER_FOLD×HPsetID with mean_*) and the raw INNER records,
	choose the optimal HPset per (Model, OUTER_FOLD) and create a list of OUTER experiment dicts.

	threshold_source: which column in summary to use for TH (usually 'mean_threshold').
	"""
	inner_df = pd.DataFrame(inner_params_records)
	winners = pick_winners(summary_df, higher_is_better=higher_is_better)

	experiments = []
	exp_id = start_exp_id

	for _, w in winners.iterrows():
		model = w['Model']
		ofold = int(w['OUTER_FOLD'])
		hpset = w['HPsetID']

		# extract hypers for this HPset
		hypers = _extract_hypers_for_hpset(inner_df, model, ofold, hpset)

		# set threshold from summary (mean across inner folds of the winning HPset)
		th = w[threshold_source] if threshold_source in w and pd.notna(w[threshold_source]) else None
		hypers['TH'] = float(th) if th is not None else None

		exp = {
			"ExpID": exp_id,
			"Model": model,
			"OUTER_FOLD": ofold,
			"hypers": hypers,
			"trained": False,
			"evaluated": False,
			"HPsetID": hpset,
		}
		experiments.append(exp)
		exp_id += 1

	return experiments


In [ ]:
outer_experiments = build_outer_experiments(
	summary_df=summary,
	inner_params_records=INNER_CV_parameters,  # raw records with 'hypers'
	higher_is_better=True,
	threshold_source='mean_threshold',         # uses the averaged inner-CV threshold per winner
	start_exp_id=1
)

len(outer_experiments)
save_dataset_info(outer_experiments, file="NCV_5_3_folds/OUTER_experiments.json")


Successfully saved data to: NCV_5_3_folds/OUTER_experiments.json


In [10]:
import itertools, random


space = {
  "orb_step":    [4, 6],
  "orb_patch":   [31],
  "vlad_k":      [64, 128],
  "desc_cap_per_case": [4000],
  "desc_cap_total":    [100000],
  "pca_dim":     [256, 512],
  "rf_n_estimators": [1000],
  "rf_max_depth": [None,20],
  "rf_min_samples_leaf": [1,2],
  "rf_max_features": ["sqrt", 0.5],
  "rf_bootstrap": [True]
}


def create_combinations(space):
	# Get the keys and the lists of values from the input dictionary
	keys = space.keys()
	values = space.values()
	# Create the Cartesian product of the value lists
	# The '*' unpacks the list of lists into arguments for the function
	combinations_as_tuples = itertools.product(*values)
	# Use a list comprehension to convert each combination tuple into a dictionary
	list_of_combo_dicts = [dict(zip(keys, combo)) for combo in combinations_as_tuples]
	return list_of_combo_dicts

def sample_n_unique(combos, n=20, seed=42):
    random.Random(seed).shuffle(combos)
    return combos[:min(n, len(combos))]


all_combinations = create_combinations(space)
sampled_sets = sample_n_unique(all_combinations, n=20, seed=42)

print(f"number of combinations of HPs: {len(all_combinations)}")  # Total number of combinations
print(len(sampled_sets), "sets sampled.")


number of combinations of HPs: 64
20 sets sampled.


In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from core.CardiacCTdataset import *

def compute_binary_metrics(y_true, y_score, th=0.5):
    y_pred = (y_score >= th).astype(int)
    auc = roc_auc_score(y_true, y_score)
    f1  = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tnr = tn / (tn + fp + 1e-8)  # specificity
    bal_acc = 0.5 * (rec + tnr)
    return dict(auc=auc, f1=f1, precision=prec, recall=rec, sensitivity=rec, specificity=tnr, balanced_acc=bal_acc)

def append_jsonl(item, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a") as f:
        f.write(json.dumps(item) + "\n")

set_0 = all_combinations[0]
print(f"first combo: {set_0}")

OUTER_FOLDS = 5; INNER_FOLDS = 3

current_out_fold = 0
current_inner_fold = 0
full_dl = load_dataset_info(file="NCV_5_3_folds/data_info_5-3NCV.json")
main_dataset = [i for i in full_dl if i["pool"] == 'main']

CFF = ClassicalFeatureFactory(main_dataset, set_0['orb_step'],
							  set_0['orb_patch'], set_0['vlad_k'], set_0['pca_dim'] )

train_loader, val_loader, fold_artifacts = CFF.create_inner_feature_loaders(current_out_fold, current_inner_fold)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

def to_numpy(loader):
    X, y = [], []
    for batch in loader:
        X.append(batch["features"].numpy())
        y.append(batch["label"].numpy())
    return np.vstack(X), np.concatenate(y)

Xtr, ytr = to_numpy(train_loader)
Xva, yva = to_numpy(val_loader)

rf = RandomForestClassifier(
    n_estimators=set_0['rf_n_estimators'], max_depth=set_0['rf_max_depth'],
    min_samples_leaf=set_0['rf_min_samples_leaf'], max_features=set_0['rf_max_features'],
    class_weight="balanced", random_state=0, n_jobs=-1
)
rf.fit(Xtr, ytr)
pva = rf.predict_proba(Xva)[:,1]
auc = roc_auc_score(yva, pva)
f1  = f1_score(yva, (pva>=0.5).astype(int))
print("AUC:", auc, "F1:", f1)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
number of combinations of HPs: 64
first combo: {'orb_step': 4, 'orb_patch': 31, 'vlad_k': 64, 'desc_cap_per_case': 4000, 'desc_cap_total': 100000, 'pca_dim': 256, 'rf_n_estimators': 1000, 'rf_max_depth': None, 'rf_min_samples_leaf': 1, 'rf_max_features': 'sqrt', 'rf_bootstrap': True}


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np, time

def to_numpy(loader):
    X, y = [], []
    for batch in loader:
        X.append(batch["features"].numpy())
        y.append(batch["label"].numpy())
    return np.vstack(X).astype(np.float32), np.concatenate(y).astype(np.int32)

def run_one_trial(outer_id, inner_id, hp_idx, cfg, results_path):
    # Build loaders (your factory already fits VLAD+PCA on inner-train)
    CFF = ClassicalFeatureFactory(
        main_dataset,
        orb_step=cfg['orb_step'],
        orb_patch=cfg['orb_patch'],
        k=cfg['vlad_k'],
        pca_dim=cfg['pca_dim'],
        num_workers=0  # safer in notebooks on Windows
    )
    t0 = time.time()
    train_loader, val_loader, artifacts = CFF.create_inner_feature_loaders(outer_id, inner_id)
    t_codebook_pca = time.time() - t0

    Xtr, ytr = to_numpy(train_loader)
    Xva, yva = to_numpy(val_loader)

    t1 = time.time()
    rf = RandomForestClassifier(
        n_estimators=cfg['rf_n_estimators'],
        max_depth=cfg['rf_max_depth'],
        min_samples_leaf=cfg['rf_min_samples_leaf'],
        max_features=cfg['rf_max_features'],
        bootstrap=cfg['rf_bootstrap'],
        class_weight="balanced",
        random_state=0, n_jobs=-1
    )
    rf.fit(Xtr, ytr)
    t_rf = time.time() - t1

    pva = rf.predict_proba(Xva)[:, 1]
    mets = compute_binary_metrics(yva, pva, th=0.5)

    res = {
        "ExpID": f"OUT{outer_id}_IN{inner_id}_HP{hp_idx}",
        "Model": "ORB_VLAD_PCA_RF",
        "OUTER_FOLD": outer_id,
        "INNER_FOLD": inner_id,
        "HPsetID": hp_idx,
        "hypers": cfg,
        "stats": {
            "n_train": int(Xtr.shape[0]),
            "n_val": int(Xva.shape[0]),
            "pos_train": int(ytr.sum()),
            "pos_val": int(yva.sum()),
            "seed": 0
        },
        "metrics": {
            "val_auc": float(mets["auc"]),
            "val_f1": float(mets["f1"]),
            "val_sensitivity": float(mets["sensitivity"]),
            "val_specificity": float(mets["specificity"]),
            "val_precision": float(mets["precision"]),
            "val_recall": float(mets["recall"]),
            "val_balanced_acc": float(mets["balanced_acc"]),
        },
        "timing_s": {
            "fit_codebook_plus_pca": float(t_codebook_pca),
            "fit_rf": float(t_rf)
        }
    }
    append_jsonl(res, results_path)
    return res

# Example driver for one inner fold with 20 random sets
results_path = "NCV_5_3_folds/INNER_results_ORB_VLAD_RF.jsonl"
for hp_idx, cfg in enumerate(sampled_sets):
    _ = run_one_trial(current_out_fold, current_inner_fold, hp_idx, cfg, results_path)
